# FADE 示例

使用 `src/fade` 最新实现，演示两类用法：
1. **`fade()`** — 批量计算 3 张测试图的雾浓度分数，并统计推理耗时
2. **`fade_with_map()`** — 计算逐块密度图，并可视化原图与热力图

In [ ]:
from pathlib import Path
import sys
import time
import warnings

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# 确保 src/ 在 sys.path 中，使 notebook 可在任意工作目录运行
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fade import fade, fade_with_map

def load_rgb(path: Path) -> np.ndarray:
    """加载图像并强制转换为 RGB uint8 数组。"""
    return np.array(Image.open(path).convert("RGB"))

IMAGE_PATHS = [
    ROOT / "test_image/test_image1.png",
    ROOT / "test_image/test_image2.JPG",
    ROOT / "test_image/test_image3.jpg",
]

In [ ]:
## 1. 批量计算 FADE 分数

# 预热：首次调用会触发 .mat 参考数据加载与 JIT 编译缓存，不计入计时
_warmup_img = np.zeros((64, 64, 3), dtype=np.uint8)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fade(_warmup_img)

# 批量计算
results = []
for p in IMAGE_PATHS:
    if not p.exists():
        results.append((p.name, None, None, None))
        continue
    img = load_rgb(p)
    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        score = fade(img)
    elapsed = time.perf_counter() - t0
    results.append((p.name, img.shape, score, elapsed))

# 打印结果表格
header = f"{'图像文件':<22} {'尺寸 (H×W×C)':<18} {'FADE 分数':>12} {'耗时 (s)':>10}"
sep    = "─" * len(header)
print(sep)
print(header)
print(sep)
for name, shape, score, elapsed in results:
    if score is None:
        print(f"{name:<22} {'[文件不存在]':<18} {'—':>12} {'—':>10}")
    else:
        h, w, c = shape
        shape_str = f"{h}×{w}×{c}"
        print(f"{name:<22} {shape_str:<18} {score:>12.6f} {elapsed:>10.4f}")
print(sep)
print("注：FADE 分数越高表示雾越浓。")

## 2. 使用 `fade_with_map()` 可视化逐块密度图

对 3 张测试图分别计算密度图，每行展示：原图 | 密度热力图（8×8 块分辨率，双线性上采样至原图尺寸）。

In [ ]:
fig, axes = plt.subplots(
    nrows=len(IMAGE_PATHS), ncols=2,
    figsize=(12, 4.5 * len(IMAGE_PATHS)),
)

for row, p in enumerate(IMAGE_PATHS):
    ax_img, ax_map = axes[row]

    if not p.exists():
        ax_img.set_title(f"{p.name}  [文件不存在]")
        ax_img.axis("off")
        ax_map.axis("off")
        continue

    img = load_rgb(p)

    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        score, density_map = fade_with_map(img)
    elapsed = time.perf_counter() - t0

    # 将密度图上采样至原图尺寸（双线性插值），便于对照
    h, w = img.shape[:2]
    density_display = np.array(
        Image.fromarray(density_map.astype(np.float32)).resize(
            (w, h), resample=Image.BILINEAR
        )
    )

    # 原图
    ax_img.imshow(img)
    ax_img.set_title(f"{p.name}\n尺寸 {h}×{w}", fontsize=10)
    ax_img.axis("off")

    # 密度热力图
    im = ax_map.imshow(density_display, cmap="hot", vmin=0)
    ax_map.set_title(
        f"FADE 密度图  |  score = {score:.4f}  |  耗时 {elapsed:.3f}s\n"
        f"密度图分辨率: {density_map.shape[0]}×{density_map.shape[1]} 块",
        fontsize=10,
    )
    ax_map.axis("off")
    fig.colorbar(im, ax=ax_map, fraction=0.046, pad=0.04, label="雾浓度（越亮越浓）")

fig.suptitle("FADE with_map 可视化", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()